**Title: Electric Vehicle Data Analysis Assignment**


NAME: REVANTH KUMAR TELAKAPALLY



COURSE: DATA ANALYSIS (3 MONTHS) **bold text**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


SECTION 1 : Data cleaning

In [ ]:
df=pd.read_csv('/content/Electric_Vehicle_Population_Data.csv')

1.How many missing values exist in the dataset, and in which columns?

In [ ]:
missing_values= df.isnull().sum()
missing_values[missing_values >0]

2.How should missing or zero values in the Base MSRP and Electric Range columns be handled?
ans:using statistical operations i.e, mean, median and mode we can fill missing values

In [ ]:
df['Electric Range'] = df['Electric Range'].fillna(df['Electric Range'].median())

In [ ]:
df['Legislative District'] = df['Legislative District'].fillna(df['Legislative District'].median())

In [ ]:
df['County'] = df['County'].fillna(df['County'].mode()[0])

df['City'] = df['City'].fillna(df['City'].mode()[0])

df['Electric Utility'] = df['Electric Utility'].fillna(df['Electric Utility'].mode()[0])

In [ ]:
df['Postal Code'] = df['Postal Code'].fillna(df['Postal Code'].mode()[0])

In [ ]:
df['Vehicle Location'] = df['Vehicle Location'].fillna('Unckown Location')

In [ ]:
df['2020 Census Tract'] = df['2020 Census Tract'].fillna(df['2020 Census Tract'].median())

In [ ]:
df.isnull().sum()

3.Are there duplicate records in the dataset? If so, how should they be managed?
ANS: no duplictes found in data

In [ ]:
duplicates = df.duplicated().sum()
print("duplicate Rows:", duplicates)

In [ ]:
df.drop_duplicates(inplace=True)

4.How can VINs be anonymized while maintaining uniqueness?
ANS: VINs can be anonymized while maintaining uniqueness by replacing the original vin values with hashed identifiers

In [ ]:
import hashlib

# Create anonymized VIN column
df['Anonymized_VIN'] = df['VIN (1-10)'].astype(str).apply(
    lambda x: hashlib.sha256(x.encode()).hexdigest()
)

# Display result
df[['VIN (1-10)', 'Anonymized_VIN']].head()

5.How can Vehicle Location (GPS coordinates) be cleaned or converted for better readability?
ANS:BY handling missing values,removing unwanted characters,and seperating gps coordinates into latitude and longitude columns.

In [ ]:
# Fill missing values
df['Vehicle Location'] = df['Vehicle Location'].fillna('Unknown Location')

# Remove unwanted text
df['Vehicle Location'] = df['Vehicle Location'].str.replace('POINT', '', regex=False)
df['Vehicle Location'] = df['Vehicle Location'].str.replace('(', '', regex=False)
df['Vehicle Location'] = df['Vehicle Location'].str.replace(')', '', regex=False)

# Split location safely
location_split = df['Vehicle Location'].str.split(expand=True)

# Create Longitude and Latitude columns
df['Longitude'] = location_split[0]
df['Latitude'] = location_split[1]

# Display output
df[['Vehicle Location', 'Longitude', 'Latitude']].head()

SECTION 2: Data Exploration Questions:

1.What are the top 5 most common EV makes and models in the dataset

In [ ]:

top_makes = df['Make'].value_counts().head(5)

print(top_makes)

2.What is the distribution of EVs by county? Which county has the most registrations?

In [ ]:
# EV count by county
county_distribution = df['County'].value_counts()

print(county_distribution.head(10))

3.How has EV adoption changed over different model years?

In [ ]:
# Count EVs by model year
yearly_ev = df['Model Year'].value_counts().sort_index()
print(yearly_ev)

4.What is the average electric range of EVs in the dataset?

In [ ]:
# Calculate average electric range
average_range = df['Electric Range'].mean()

print("Average Electric Range:", round(average_range, 2))

5.What percentage of EVs are eligible for Clean Alternative Fuel Vehicle (CAFV) incentives?

In [ ]:
cafv_counts = df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].value_counts()
total_vehicles = len(df)
cafv_percentage = (cafv_counts / total_vehicles) * 100
print(round(cafv_percentage, 2))

6.How does the electric range vary across different makes and models?

In [ ]:
make_range = df.groupby('Make')['Electric Range'].mean().sort_values(ascending=False).head(10)
print(make_range)

7.What is the average Base MSRP for each EV model?

In [ ]:
print(df.columns.tolist())


In [ ]:
avg_msrp_model = df.groupby('Model')['Base MSRP'].mean

8.Are there any regional trends in EV adoption (e.g., urban vs. rural areas)?

In [ ]:
city_distribution = df['City'].value_counts().head(15)

print(city_distribution)

SECTION 3. Data Visualization Questions:

1.Create a bar chart showing the top 5 EV makes and models by count

In [ ]:
top_makes = df['Make'].value_counts().head(5)

top_models = df['Model'].value_counts().head(5)

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)

top_makes.plot(kind='bar')

plt.title('Top 5 EV Makes')
plt.xlabel('Make')
plt.ylabel('Count')

plt.xticks(rotation=45)

plt.subplot(1,2,2)

top_models.plot(kind='bar')

plt.title('Top 5 EV Models')
plt.xlabel('Model')
plt.ylabel('Count')

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

2.Use a heatmap or choropleth map to visualize EV distribution by county.

In [ ]:
county_counts = df['County'].value_counts().head(20)

plt.figure(figsize=(12,8))

sns.heatmap(
    county_counts.to_frame(),
    annot=True,
    fmt='g',
    cmap='YlGnBu'
)

plt.title('EV Distribution by County')

plt.show()

3.Create a line graph showing the trend of EV adoption by model year

In [ ]:
yearly_ev = df['Model Year'].value_counts().sort_index()

plt.figure(figsize=(12,6))

plt.plot(
    yearly_ev.index,
    yearly_ev.values,
    marker='o'
)

plt.title('EV Adoption Trend by Model Year')
plt.xlabel('Model Year')
plt.ylabel('Number of EVs')

plt.grid(True)

plt.show()

4.Generate a scatter plot comparing electric range vs. base MSRP to see pricing trends

5.Plot a pie chart showing the proportion of CAFV-eligible vs. non-eligible EVs.

In [ ]:
cafv_counts = df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].value_counts()

plt.figure(figsize=(8,8))

plt.pie(
    cafv_counts,
    labels=cafv_counts.index,
    autopct='%1.1f%%'
)

plt.title('CAFV Eligibility Proportion')

plt.show()

6.Use a geospatial map to display EV registrations based on vehicle location.

In [ ]:
import folium

location_data = df.dropna(subset=['Latitude', 'Longitude'])

location_data['Latitude'] = pd.to_numeric(location_data['Latitude'], errors='coerce')
location_data['Longitude'] = pd.to_numeric(location_data['Longitude'], errors='coerce')

location_data = location_data.dropna(subset=['Latitude', 'Longitude'])

ev_map = folium.Map(
    location=[location_data['Latitude'].mean(),
              location_data['Longitude'].mean()],
    zoom_start=7
)

for _, row in location_data.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=2
    ).add_to(ev_map)

ev_map

SECTION 4 : Linear Regression Model Questions:

1.How can we use Linear Regression to predict the Electric Range of a vehicle?
ANA:Linear Regression can be used to predict the Electric Range of a vehicle by identifying relationships between the target variable and independent features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

ml_df = df[['Model Year', 'Make', 'Electric Range']]

ml_df = ml_df.dropna()

encoder = LabelEncoder()

ml_df['Make'] = encoder.fit_transform(ml_df['Make'])

X = ml_df[['Model Year',  'Make']]

y = ml_df['Electric Range']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

score = r2_score(y_test, y_pred)

print("R2 Score:", score)

2.What independent variables (features) can be used to predict Electric Range? (e.g., Model Year, Base MSRP, Make)
ANS:Independent variables that can be used to predict the Electric Range of a vehicle include Model Year, Base MSRP, Make, Model, Electric Vehicle Type, and CAFV Eligibility. Model Year helps capture technological advancements in battery performance over time, while Base MSRP reflects vehicle pricing and premium features that may influence range. Manufacturer and model information help identify differences in battery efficiency across brands. Electric Vehicle Type and CAFV Eligibility also provide useful insights into vehicle performance and regulatory classification.


3.How do we handle categorical variables like Make and Model in regression analysis?
ANS:Categorical variables such as Make and Model cannot be used directly in regression analysis because machine learning algorithms work with numerical data. To handle this, the categorical values are converted into numeric representations using encoding techniques. One common approach is Label Encoding, where each unique category is assigned a numerical value. Another method is One-Hot Encoding, which creates separate binary columns for each category. This allows the regression model to process manufacturer and model information effectively while identifying patterns related to electric range prediction.


4.What is the R² score of the model, and what does it indicate about prediction accuracy?
ANS:The R² score measures how well the regression model explains the variation in the target variable, which in this case is Electric Range. Its value ranges from 0 to 1. A score closer to 1 indicates that the model predicts the electric range more accurately, while a score closer to 0 means the model does not explain the data well. For example, an R² score of 0.80 means that the model can explain 80% of the variation in electric range using the selected features. This metric helps evaluate the overall prediction performance of the regression model.


5.How does the Base MSRP influence the Electric Range according to the regression model?
ANS:According to the regression model, Base MSRP shows a positive relationship with Electric Range. This means that vehicles with higher prices generally tend to offer longer electric ranges. Higher MSRP vehicles often include larger battery capacities, advanced technology, and better energy efficiency, which contribute to improved driving range. The regression analysis helps identify this trend by measuring how changes in vehicle price are associated with changes in electric range.


6.What steps are needed to improve the accuracy of the Linear Regression model?
ANS:Handle missing values properly,
Remove duplicate records,
Detect and treat outliers,
Add more relevant features,
Encode categorical ,variables correctly,
Perform feature scaling if required,
Increase training data quality and size,
Select the most important features,
Reduce irrelevant or noisy data,
Try advanced models like Random Forest or XGBoost for better performance

7.Can we use this model to predict the range of new EV models based on their specifications?
ANS:Yes, the Linear Regression model can be used to predict the electric range of new EV models based on their specifications. By providing input features such as Model Year, Base MSRP, Make, and other relevant vehicle details, the model can estimate the expected electric range. The prediction accuracy depends on the quality of training data, selected features, and how well the new vehicle specifications match the patterns learned from the existing dataset.


KEY FINDINGS AND ANALYSIS OVERVIEW

This project analyzed Electric Vehicle population data using Python and Power BI to understand EV adoption trends, vehicle distribution, and market insights. The dataset was cleaned by handling missing values, removing duplicates, correcting data types, and improving location readability. Exploratory Data Analysis (EDA) and visualizations were performed to identify important patterns in the data.
A Linear Regression model was also developed to predict Electric Range using features such as Model Year and Manufacturer. Interactive dashboards were created in Power BI to present insights effectively through charts, maps, and KPI cards.
Key Findings and Insights
Electric Vehicle adoption has increased significantly in recent years, especially after 2020.
Tesla emerged as the leading EV manufacturer with the highest number of registrations.
Battery Electric Vehicles (BEVs) are more common than Plug-in Hybrid Electric Vehicles (PHEVs).
Certain cities and counties showed higher EV concentration, indicating stronger regional adoption.
Most EVs fall within a moderate electric range category, while newer models provide higher ranges.
CAFV eligibility plays an important role in EV adoption and government incentive distribution.
Data preprocessing and visualization improved overall understanding of EV market trends.
The Linear Regression model demonstrated that features like Model Year and Make influence Electric Range prediction.
Overall, the project provided valuable insights into the growing Electric Vehicle ecosystem and demonstrated practical applications of data cleaning, visualization, machine learning, and business intelligence tools.